# 08 · SFT on Stockfish's moves (`sft_stockfish`)

**The question.** Our first SFT model (`sft_v1`, notebook 03) learned to copy the move a strong human player (rated 2000+) made in each position. A puzzle, however, accepts only the single best move. This experiment asks: **does SFT learn more if each answer is Stockfish's best move instead of the human's move?** Our TA suggested this ablation.

**What changes, and what stays the same.** Only the answer the model learns changes. Everything else is identical to `sft_v1`, so a difference in results can be attributed to the answer source:

| | `sft_v1` (notebook 03) | `sft_stockfish` (this notebook) |
|---|---|---|
| Training positions | 20,000 from 2000+ games | **the same 20,000**, in the same order |
| Answer the model learns | the move the player made | **Stockfish's best move** (search depth 10) |
| Model, LoRA, learning rate, epochs, seed | `configs/sft.yaml` | the same |
| Graded on | 1,000 test puzzles | the same 1,000 puzzles, same grader |

Depth 10 is the depth already used for the "chosen" moves of DPO, so after this change SFT, DPO, and RL all learn from the same engine.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run.
3. Set `QUICK_CHECK = False` and run the real thing. For a run this long, use a **Job** so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification.
4. Afterwards, commit `results/` from the Git folder; Databricks cannot commit notebook outputs. Then **File → Export → IPython Notebook**, replace this notebook with the exported file on a laptop, and commit it there, so the figures also show inside the notebook on GitHub.

**Time:** labelling the positions with Stockfish takes a few minutes (only once; the result is saved), training about 1 hour on an A10, grading about 5 minutes.

**Outputs**
- the adapter in `checkpoints/sft_stockfish/` (not on GitHub)
- results: `results/sft_stockfish_training.json`, `results/sft_stockfish_dev.json`, `results/sft_stockfish.json`, and a new row in `results/summary.csv`
- every figure and table of this experiment in `results/figures/sft_stockfish/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
RUN_NAME = "sft_stockfish"
QUICK_CHECK = True    # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.

import gc

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.data import add_stockfish_moves
from chesslm.evaluate import load_eval_model
from chesslm.experiments import compare_with, figures_dir, grade_puzzles, puzzle_table, save_table, save_training
from chesslm.train_sft import sample_examples, to_dataset, train

sft_cfg = load_config("sft")
sft_cfg["data"]["answer"] = "stockfish"   # THE change in this experiment; everything else is as in sft_v1
prompt_cfg, eval_cfg = load_config("prompt"), load_config("eval")
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
SEED = sft_cfg["training"]["seed"]
LABEL_WORKERS = 4   # Stockfish processes running at once while labelling (fixed, so labels are reproducible)

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUN_NAME += "_check"
    sft_cfg["data"].update(trial_examples=16, train_examples=64, val_examples=16)
    sft_cfg["training"].update(logging_steps=1, eval_steps=1)
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
ADAPTER_DIR = REPO_ROOT / "checkpoints" / RUN_NAME
FIGURES = figures_dir(RUN_NAME)
print(f"Run: {RUN_NAME} | answer learned: {sft_cfg['data']['answer']} | figures and tables -> {FIGURES}")

## 1. Training data: the same positions, with Stockfish's move as the answer

We take exactly the positions `sft_v1` trained on (the same seeded sample of 20,000 training and 1,000 validation positions) and ask Stockfish for its best move in each. The labelled positions are saved in `data/processed/`, so this step runs only once.

In [ ]:
PROCESSED = REPO_ROOT / "data" / "processed"
n = sft_cfg["data"]
label_cfg = {**eval_cfg["stockfish"], "depth": n["stockfish_depth"]}

labelled = {}
for split, size in [("train", n["train_examples"]), ("val", n["val_examples"])]:
    sample = sample_examples(pd.read_parquet(PROCESSED / f"{split}.parquet"), size, SEED)  # the sft_v1 positions
    path = PROCESSED / f"sft_stockfish_{split}.parquet"
    if path.exists() and not QUICK_CHECK:
        labelled[split] = pd.read_parquet(path)
        assert list(labelled[split].fen) == list(sample.fen), f"{path.name} holds other positions; delete it and rerun"
        print(f"{split}: loaded {len(labelled[split]):,} labelled positions from {path.name}")
    else:
        print(f"{split}: asking Stockfish for the best move in {len(sample):,} positions ...")
        labelled[split] = add_stockfish_moves(sample, label_cfg, LABEL_WORKERS)
        if not QUICK_CHECK:
            labelled[split].to_parquet(path)

How different are the two kinds of answers? If the players already chose Stockfish's move most of the time, this experiment changes little. The table below gives the share of training positions where the human move and Stockfish's move are the same, and three examples where they differ.

In [ ]:
train_rows = labelled["train"]
same = (train_rows.move_uci == train_rows.best_uci).mean()
data_table = pd.DataFrame({
    "Training positions": [len(train_rows)],
    "Human move = Stockfish's move (%)": [round(same * 100, 1)],
    "Answer changed by this experiment (%)": [round((1 - same) * 100, 1)],
}, index=pd.Index(["sft_stockfish training data"], name="Data"))
save_table(data_table, FIGURES, "training_data",
           "Share of the 20,000 training positions where the move played by the 2000+ player equals Stockfish's best move (depth 10).")
display(data_table)

print("Examples where they differ:")
for row in train_rows[train_rows.move_uci != train_rows.best_uci].head(3).itertuples():
    print(f"  {row.fen}\n    player: {row.move_san:<8} Stockfish: {row.best_san}")

## 2. Trial run

A short run on the first few hundred training examples, to check that the loss goes down on this machine before the long run.

In [ ]:
trial_ds = to_dataset(train_rows.head(n["trial_examples"]), prompt_cfg, answer="stockfish")
val_ds = to_dataset(labelled["val"], prompt_cfg, answer="stockfish")
example = trial_ds[0]
print("Prompt:    ", example["prompt"][0]["content"].replace("\n", "\n            "))
print("Completion:", example["completion"][0]["content"], "\n")

trial_log, trial_compute = train(trial_ds, val_ds.select(range(min(100, len(val_ds)))), sft_cfg,
                                 REPO_ROOT / "checkpoints" / f"{RUN_NAME}_trial")
losses = trial_log.loss.dropna()
print(f"\nTraining loss: {losses.iloc[0]:.3f} at the start -> {losses.iloc[-1]:.3f} at the end")
gc.collect()
torch.cuda.empty_cache()

## 3. Full training run

The same settings as `sft_v1`. The figure shows the loss on training examples and on the 1,000 held-out validation positions; both should fall and level off.

In [ ]:
train_ds = to_dataset(train_rows, prompt_cfg, answer="stockfish")
print(f"train: {len(train_ds):,} examples  validation: {len(val_ds):,} examples")
log, compute = train(train_ds, val_ds, sft_cfg, ADAPTER_DIR)
save_training(RUN_NAME, {"sft": sft_cfg, "prompt": prompt_cfg}, compute, log)
plots.loss_curve(log, FIGURES / "training_loss.png", title=f"{RUN_NAME}: loss during training");

## 4. Grade the model on the puzzles

Exactly the grader, puzzles, and greedy decoding used for every other model. The 200 dev puzzles come first as a quick check, then the 1,000 test puzzles.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=ADAPTER_DIR)
grade_puzzles(model, tokenizer, "dev", f"{RUN_NAME}_dev", configs, str(ADAPTER_DIR), PUZZLE_LIMIT)
grade_puzzles(model, tokenizer, "test", RUN_NAME, configs, str(ADAPTER_DIR), PUZZLE_LIMIT);

## 5. Results: base model vs. `sft_v1` vs. `sft_stockfish`

**How to read this section**
- **Puzzle accuracy** is the main metric: the share of the 1,000 test puzzles solved. For reference, a random legal move solves 5.6%.
- **Legal / illegal / unparseable** show whether the answers are valid moves at all.
- **ACPL** (average centipawn loss) is how much worse the legal answers are than Stockfish's best move, in hundredths of a pawn; lower is better.
- The **paired comparison** counts the puzzles that only one of two models solves. A p-value below 0.05 means the difference is unlikely to be chance.

All numbers are on the 1,000 test puzzles.

In [ ]:
runs = ["base", "sft_v1", RUN_NAME]
display(puzzle_table(runs, FIGURES))

comparison = pd.DataFrame([compare_with(RUN_NAME, "sft_v1"), compare_with(RUN_NAME, "base")]).set_index("model")
save_table(comparison, FIGURES, "paired_comparison",
           "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
display(comparison)

In [ ]:
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
plots.compare_metrics(summary, runs, FIGURES / "puzzle_metrics.png",
                      title="Human moves (sft_v1) vs. Stockfish's moves (sft_stockfish) as SFT answers");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png",
                       title="Puzzle accuracy by puzzle difficulty (rating band)");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png",
                        title="Are the answers legal moves?");

## Summary of outputs

- `results/figures/sft_stockfish/`: `training_data` (how often the human and Stockfish moves agree), `training_loss.png`, `puzzle_results` (table), `paired_comparison` (table), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/sft_stockfish.json`: every test answer and its grade; `results/sft_stockfish_training.json`: settings, compute, training log
- The model is also graded on held-out game positions and in full games in notebook 11.
- If `sft_stockfish` beats `sft_v1`, it becomes the starting model for the combinations in notebook 12.